In [30]:
import os
import re
import json
import uuid
import math
from typing import List, Dict, Any, Optional
from tqdm import tqdm
from dotenv import load_dotenv

import pdfplumber
import numpy as np

from langchain_groq import ChatGroq
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

load_dotenv()

True

In [31]:
os.chdir("/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject")
os.getcwd()

'/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject'

In [32]:
from src.Config.config import NEO4J_PASSWORD , NEO4J_URI , NEO4J_USERNAME , EMBEDDING_MODEL
from src.LLMs.Qwen_Model import ArabicModel

# embeddings
EMBED_MODEL = EMBEDDING_MODEL

# Hyperparams for hybrid scoring
WEIGHT_VECTOR = 0.5
WEIGHT_PAGERANK = 0.3
WEIGHT_CONCEPT_OVERLAP = 0.2

TOP_K = 10

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

armodel = ArabicModel()
llm = armodel.get_ar_llm()

In [ ]:
# ---------- UTILITIES ----------
def normalize_arabic(text: str) -> str:
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('ـ', '')
    return text.strip()

def safe_json_load(s: str) -> Any:
    s = s.strip()
    start = s.find('{')
    end = s.rfind('}')
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(s[start:end+1])
        except Exception:
            pass
    try:
        return json.loads(s)
    except Exception:
        raise ValueError("Failed to parse JSON from extractor output.")


def cosine_sim(a: List[float], b: List[float]) -> float:
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# ---------- PDF LOADING & PARSING ----------
def load_pdfs(folder: str) -> List[Dict[str, Any]]:
    docs = []
    files = sorted([f for f in os.listdir(folder) if f.lower().endswith((".pdf", ".txt"))])
    
    for file in files:
        path = os.path.join(folder, file)
        text = ""
        
        if file.lower().endswith(".pdf"):
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    t = page.extract_text()
                    if t:
                        text += t + "\n"
        elif file.lower().endswith(".txt"):
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
        
        docs.append({
            "law_name": file.replace(".pdf", "").replace(".txt", ""),
            "raw_text": normalize_arabic(text)
        })
        
    return docs


def split_articles(doc: Dict[str, Any]) -> List[Dict[str, Any]]:
    text = doc["raw_text"]
    parts = re.split(r'((?:م|المادة)\s*\d+[^\n]*)', text)
    articles = []
    if len(parts) >= 3:
        for i in range(1, len(parts), 2):
            header = parts[i].strip()
            content = parts[i+1].strip() if i+1 < len(parts) else ""
            article_id_match = re.search(r'\d+', header)
            article_id = article_id_match.group(0) if article_id_match else header
            articles.append({
                "law": doc["law_name"],
                "article_id": f"{doc['law_name']}_article_{article_id}",
                "article_number": article_id,
                "text": content
            })
    else:
        articles.append({
            "law": doc["law_name"],
            "article_id": f"{doc['law_name']}_full_1",
            "article_number": None,
            "text": text
        })
    return articles

In [34]:
# ---------- EXTRACTION (LLM or heuristic) ----------
LLM_PROMPT_TEMPLATE = """
You are a JSON generator. Given this Arabic legal article text, return a JSON with exactly this schema:

{{
  "rules": [
    {{
      "rule_id": "<id-string>",
      "rule_text": "<extracted rule text in Arabic>",
      "concepts": ["<list of short Arabic concept strings>"],
      "penalty": {{
          "type": "<سجن/حبس/غرامة/سجن مؤبد/etc or null>",
          "min_value": "<min years or null>",
          "max_value": "<max years or null>",
          "notes":"<any textual penalty note>"
      }}
    }}
  ]
}}

Text:
{article_text}

Return JSON only.
"""

CRIME_KEYWORDS = [
    "قتل","سرقة","القتل","الحرابة","التهرب","الاحتيال","مخدرات","إتجار","تعاطي",
    "غسيل أموال","أسلحة","ذخيرة","اعتداء","اغتصاب","اختطاف","رشوة","تبديد","تزوير",
    "جرائم إلكترونية","حيازة"
]

PENALTY_KEYWORDS = ["سجن","حبس","غرامة","مؤبد","إعدام","سنوات","شهر","أشهر","يوم"]

def extract_rules_openai(article_text: str) -> Dict[str, Any]:    
    prompt = LLM_PROMPT_TEMPLATE.format(article_text=article_text)
    resp = llm.invoke(prompt)
    out = resp["message"][-1]["content"]
    return safe_json_load(out)


def extract_rules_heuristic(article_text: str) -> Dict[str, Any]:
    sentences = re.split(r'(?<=[\.؟!\n])\s+', article_text)
    rules = []
    for s in sentences:
        s_clean = s.strip()
        if not s_clean:
            continue
        if any(k in s_clean for k in CRIME_KEYWORDS) or any(p in s_clean for p in PENALTY_KEYWORDS):
            concepts = [kw for kw in CRIME_KEYWORDS if kw in s_clean]
            penalty_type = None
            min_v = None
            max_v = None
            notes = ""
            pen_match = re.search(r'(\d+)\s*(سنة|سنوات|شهر|أشهر|يوم|أيام)', s_clean)
            if "سجن مؤبد" in s_clean or "سجنٍ مؤبد" in s_clean:
                penalty_type = "سجن مؤبد"
            elif "سجن" in s_clean:
                penalty_type = "سجن"
            elif "حبس" in s_clean:
                penalty_type = "حبس"
            elif "غرامة" in s_clean:
                penalty_type = "غرامة"
            if pen_match:
                val = int(pen_match.group(1))
                min_v = val
                max_v = val
            notes = s_clean if penalty_type else ""
            rules.append({
                "rule_id": str(uuid.uuid4()),
                "rule_text": s_clean,
                "concepts": [c for c in concepts] if concepts else [],
                "penalty": {
                    "type": penalty_type,
                    "min_value": min_v,
                    "max_value": max_v,
                    "notes": notes
                }
            })
    if not rules:
        rules.append({
            "rule_id": str(uuid.uuid4()),
            "rule_text": article_text.strip()[:2000],
            "concepts": [],
            "penalty": {"type": None, "min_value": None, "max_value": None, "notes": ""}
        })
    return {"rules": rules}


def extract_rules(article_text: str) -> Dict[str, Any]:
    if True:
        try:
            return extract_rules_openai(article_text)
        except Exception as e:
            print("OpenAI extractor failed, falling back to heuristic. Error:", e)
            return extract_rules_heuristic(article_text)
    else:
        return extract_rules_heuristic(article_text)

In [35]:
# ---------- EMBEDDINGS ----------
def embed_text(text: str) -> List[float]:
    v = EMBED_MODEL.encode(text)
    return v.tolist()


# ---------- NEO4J INSERTION ----------
def create_constraints():
    with driver.session() as s:
        s.run("CREATE CONSTRAINT IF NOT EXISTS FOR (l:Law) REQUIRE l.name IS UNIQUE")
        s.run("CREATE CONSTRAINT IF NOT EXISTS FOR (a:Article) REQUIRE a.article_id IS UNIQUE")
        s.run("CREATE CONSTRAINT IF NOT EXISTS FOR (r:LegalRule) REQUIRE r.rule_id IS UNIQUE")
        s.run("CREATE CONSTRAINT IF NOT EXISTS FOR (c:LegalConcept) REQUIRE c.name IS UNIQUE")


def insert_law_and_article(article: Dict[str, Any]):
    with driver.session() as s:
        s.run(
            """
            MERGE (l:Law {name:$law})
            MERGE (a:Article {article_id:$article_id})
            SET a.text = $text, a.article_number = $article_number
            MERGE (l)-[:HAS_ARTICLE]->(a)
            """,
            {
                "law": article["law"],
                "article_id": article["article_id"],
                "text": article["text"],
                "article_number": article.get("article_number")
            }
        )


def insert_rule_and_concepts(article_id: str, rule: Dict[str, Any]):
    # rule contains: rule_id, rule_text, concepts, penalty
    embedding = embed_text(rule["rule_text"])
    with driver.session() as s:
        s.run(
            """
            MATCH (a:Article {article_id:$article_id})
            MERGE (r:LegalRule {rule_id:$rule_id})
            SET r.text = $text, r.embedding = $embedding, r.penalty = $penalty
            MERGE (a)-[:CONTAINS_RULE]->(r)
            """,
            {
                "article_id": article_id,
                "rule_id": rule["rule_id"],
                "text": rule["rule_text"],
                "embedding": embedding,
                "penalty": json.dumps(rule.get("penalty", {}))
            }
        )
        # concepts
        for c in rule.get("concepts", []):
            cname = c.strip()
            if not cname:
                continue
            s.run(
                """
                MATCH (r:LegalRule {rule_id:$rule_id})
                MERGE (c:LegalConcept {name:$cname})
                MERGE (r)-[:RELATES_TO]->(c)
                """,
                {"rule_id": rule["rule_id"], "cname": cname}
            )


# ---------- GDS / PAGERANK ----------
def compute_pagerank_graph(project_name: str = "legalGraph"):
    with driver.session() as s:
        try:
            s.run(
                """
                CALL gds.graph.project(
                  $graph_name,
                  ['LegalRule','LegalConcept'],
                  { RELATES_TO: { } }
                )
                """,
                {"graph_name": project_name}
            )
            s.run(
                """
                CALL gds.pageRank.write($graph_name, {writeProperty: 'pagerank'})
                """,
                {"graph_name": project_name}
            )
        except Exception as e:
            print("GDS operations failed — تأكد من وجود GDS plugin في Neo4j. Error:", e)


# ---------- RETRIEVAL & HYBRID RANKING ----------
def get_candidate_rules_by_concepts(concepts: List[str], limit=200) -> List[Dict[str, Any]]:
    # fetch rules that relate to any of the concepts
    with driver.session() as s:
        res = s.run(
            """
            MATCH (r:LegalRule)-[:RELATES_TO]->(c:LegalConcept)
            WHERE c.name IN $concepts
            RETURN DISTINCT r.rule_id AS rule_id, r.text AS text, r.embedding AS embedding, r.penalty AS penalty, r.pagerank AS pagerank
            LIMIT $limit
            """,
            {"concepts": concepts, "limit": limit}
        )
        rows = []
        for r in res:
            emb = r["embedding"]
            # Neo4j may store JSON string for penalty
            pen = r["penalty"]
            try:
                pen = json.loads(pen) if isinstance(pen, str) else pen
            except Exception:
                pen = pen
            rows.append({
                "rule_id": r["rule_id"],
                "text": r["text"],
                "embedding": emb,
                "penalty": pen,
                "pagerank": float(r["pagerank"] or 0.0)
            })
        return rows


def get_top_rules_by_hybrid(case_text: str, topk: int = TOP_K) -> List[Dict[str, Any]]:
    # 1) extract concepts from case_text
    if True:
        prompt = f"استخراج قائمة قصيرة من المفاهيم القانونية (عربية، كلمات/عبارات قصيرة) من النص التالي:\n\n{case_text}\n\nأعد JSON: {{\"concepts\": [..]}}"
        try:
            resp = llm.invoke(prompt)
            out = resp["message"][-1]["content"]
            j = safe_json_load(out)
            concepts = j.get("concepts", [])
        except Exception:
            # fallback heuristic
            concepts = [kw for kw in CRIME_KEYWORDS if kw in case_text]
    else:
        concepts = [kw for kw in CRIME_KEYWORDS if kw in case_text]

    # 2) embed case
    case_emb = embed_text(case_text)

    # 3) get candidates
    candidates = []
    if concepts:
        candidates = get_candidate_rules_by_concepts(concepts, limit=500)

    # if no candidates found, fetch a sample of rules
    if not candidates:
        with driver.session() as s:
            res = s.run(
                """
                MATCH (r:LegalRule)
                RETURN r.rule_id AS rule_id, r.text AS text, r.embedding AS embedding, r.penalty AS penalty, r.pagerank AS pagerank
                LIMIT 500
                """
            )
            for r in res:
                emb = r["embedding"]
                pen = r["penalty"]
                try:
                    pen = json.loads(pen) if isinstance(pen, str) else pen
                except Exception:
                    pen = pen
                candidates.append({
                    "rule_id": r["rule_id"],
                    "text": r["text"],
                    "embedding": emb,
                    "penalty": pen,
                    "pagerank": float(r["pagerank"] or 0.0)
                })

    # 4) score candidates: vector_similarity, pagerank(normalized), concept_overlap
    # normalize pagerank
    pageranks = [c.get("pagerank", 0.0) for c in candidates]
    max_pr = max(pageranks) if pageranks else 1.0
    scored = []
    for c in candidates:
        emb = c.get("embedding") or []
        vec_sim = cosine_sim(case_emb, emb) if emb else 0.0
        pr_norm = (c.get("pagerank", 0.0) / max_pr) if max_pr > 0 else 0.0
        # concept overlap: count how many concepts present in rule.text
        overlap = 0
        if concepts:
            overlap = sum(1 for cc in concepts if cc in (c.get("text") or ""))
            overlap = overlap / len(concepts)
        score = WEIGHT_VECTOR * vec_sim + WEIGHT_PAGERANK * pr_norm + WEIGHT_CONCEPT_OVERLAP * overlap
        scored.append({**c, "vec_sim": vec_sim, "pr_norm": pr_norm, "overlap": overlap, "score": score})

    # sort
    scored_sorted = sorted(scored, key=lambda x: x["score"], reverse=True)
    return scored_sorted[:topk]

In [36]:
from tqdm import tqdm
from typing import List, Dict, Any, Optional, Tuple
# PDF loader
import fitz  # PyMuPDF

# embeddings
from sentence_transformers import SentenceTransformer

# Document class (LangChain)
try:
    from langchain_classic.schema import Document
except Exception:
    # fallback simple dataclass if langchain not present
    from dataclasses import dataclass

    @dataclass
    class Document:
        page_content: str
        metadata: Dict[str, Any]


# ---------- 1) Arabic normalization ----------
def normalize_arabic(text: str) -> str:
    """
    Basic Arabic normalization to remove weird unicode chars and normalize
    common letter variants. Tailor to your needs.
    """
    if text is None:
        return ""
    # remove zero-width and other weird chars (common in scraped PDFs)
    text = re.sub(r"[\u200c\u200b\u202c\u200e\u200f]", "", text)
    # basic letter normalization
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا").replace("ى", "ي")
    text = text.replace("ؤ", "و").replace("ئ", "ي").replace("ة", "ه")
    # collapse multiple spaces and trim
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ---------- 2) Low-level loaders ----------
def load_pdf_pymupdf(pdf_path: str) -> Optional[str]:
    """Load PDF content using PyMuPDF (fitz). Returns raw text or None on error."""
    try:
        doc = fitz.open(pdf_path)
        text_pages = []
        for page in doc:
            # get_text() returns text; fall back to get_text("text") if needed
            t = page.get_text()
            if t:
                text_pages.append(t)
        doc.close()
        return "\n".join(text_pages)
    except Exception as e:
        print(f"Error loading PDF {pdf_path}: {e}")
        return None


def load_text_file(txt_path: str) -> Optional[str]:
    """Load text file content safely (utf-8, ignore errors)."""
    try:
        with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    except Exception as e:
        print(f"Error loading text file {txt_path}: {e}")
        return None


# ---------- 3) Document-level loader (folder) ----------
def load_files_from_folder(folder: str,
                           allowed_ext: Tuple[str, ...] = (".pdf", ".txt"),
                           normalize: bool = True,
                           category: Optional[str] = None) -> List[Document]:
    """
    Load all supported files in a folder and return a list of Document objects.
    `category` will be stored in metadata if provided.
    """
    documents: List[Document] = []
    files = sorted([f for f in os.listdir(folder) if f.lower().endswith(allowed_ext)])
    for file in files:
        full_path = os.path.join(folder, file)
        ext = os.path.splitext(file)[1].lower()
        raw_text = None

        if ext == ".pdf":
            raw_text = load_pdf_pymupdf(full_path)
        elif ext == ".txt":
            raw_text = load_text_file(full_path)
        else:
            print(f"⚠ Skipping unsupported file type: {file}")
            continue

        if raw_text is None:
            print(f"✗ Failed to load: {file}")
            continue

        if normalize:
            raw_text = normalize_arabic(raw_text)

        metadata = {"source": file, "path": full_path}
        if category:
            metadata["category"] = category

        documents.append(Document(page_content=raw_text, metadata=metadata))
        print(f"✓ Loaded: {file} (chars: {len(raw_text)})")

    return documents


# ---------- 4) (optional) simple chunker ----------
def chunk_text(text: str, max_chars: int = 2000, overlap: int = 200) -> List[str]:
    """
    Naive chunker that splits text into slices roughly `max_chars` long with `overlap`.
    Works by splitting on newlines when possible.
    """
    if not text:
        return []
    paragraphs = text.split("\n")
    chunks = []
    current = ""
    for par in paragraphs:
        par = par.strip()
        if not par:
            continue
        if len(current) + len(par) + 1 <= max_chars:
            current = (current + "\n" + par).strip()
        else:
            if current:
                chunks.append(current)
            # if single paragraph longer than max_chars, split it forcibly
            if len(par) > max_chars:
                for i in range(0, len(par), max_chars - overlap):
                    chunks.append(par[i:i + (max_chars - overlap)])
                current = ""
            else:
                current = par
    if current:
        chunks.append(current)
    return chunks


# ---------- 5) Embedding helpers ----------
class Embedder:
    def __init__(self, model_name: str = "silma-ai/silma-embeddding-sts-v0.1", device: str = "cpu"):
        """
        Initialize SentenceTransformer embedder. Change device if GPU available (e.g., 'cuda').
        """
        print(f"Loading embedding model: {model_name} (device={device}) ...")
        self.model = SentenceTransformer(model_name, device=device)
        print("Model loaded.")

    def embed_texts(self, texts: List[str], batch_size: int = 32) -> List[List[float]]:
        """
        Embed a list of texts and return list of vectors (lists of floats).
        """
        # ensure strings
        texts = [t if isinstance(t, str) else "" for t in texts]
        embs = self.model.encode(texts, batch_size=batch_size, show_progress_bar=True)
        # convert to plain python lists
        return [e.tolist() for e in embs]


# ---------- 6) High-level pipeline ----------
def build_document_index(base_folder: str,
                         embed_model_name: Optional[str] = None,
                         chunk_long_docs: bool = True,
                         chunk_size: int = 2000,
                         chunk_overlap: int = 200,
                         embed_device: str = "cpu") -> Dict[str, Any]:
    """
    Walks `base_folder`, loads documents (subfolders treated as categories), optionally chunks,
    and returns a dictionary with:
        - documents: list of Document objects (either full or chunk-level)
        - embeddings: list of vectors (if embed_model_name provided; otherwise empty)
        - metadata: parallel list of metadata dicts
    """
    all_documents: List[Document] = []
    # if base_folder contains many files directly, treat base_folder as a single category
    entries = sorted(os.listdir(base_folder))
    # if there are subfolders, iterate each; else treat base_folder as single folder
    subfolders = [e for e in entries if os.path.isdir(os.path.join(base_folder, e))]
    if subfolders:
        for sub in subfolders:
            folder_path = os.path.join(base_folder, sub)
            print(f"📂 Loading folder/category: {sub}")
            docs = load_files_from_folder(folder_path, category=sub)
            all_documents.extend(docs)
    else:
        print(f"📂 Loading files from base folder: {base_folder}")
        all_documents.extend(load_files_from_folder(base_folder, category=None))

    processed_docs: List[Document] = []
    for doc in all_documents:
        text = doc.page_content or ""
        if chunk_long_docs:
            chunks = chunk_text(text, max_chars=chunk_size, overlap=chunk_overlap)
            if len(chunks) <= 1:
                processed_docs.append(doc)
            else:
                # create a Document per chunk, preserving metadata and adding chunk index
                for i, chunk in enumerate(chunks):
                    md = dict(doc.metadata)
                    md["chunk_index"] = i
                    md["chunk_total"] = len(chunks)
                    processed_docs.append(Document(page_content=chunk, metadata=md))
        else:
            processed_docs.append(doc)

    print(f"✅ Total processed document objects (after chunking): {len(processed_docs)}")

    result = {"documents": processed_docs, "embeddings": [], "metadata": [d.metadata for d in processed_docs]}

    # embed if model provided
    if embed_model_name:
        embedder = Embedder(embed_model_name, device=embed_device)
        texts = [d.page_content for d in processed_docs]
        print("🔢 Embedding texts ...")
        embeddings = embedder.embed_texts(texts)
        result["embeddings"] = embeddings
        print(f"✅ Embedded {len(embeddings)} texts (vectors length = {len(embeddings[0]) if embeddings else 0})")

    return result

import pickle
# ---------- 7) Save / Load helpers ----------
def save_index(obj: Dict[str, Any], out_path: str):
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved index to: {out_path}")


def load_index(in_path: str) -> Dict[str, Any]:
    with open(in_path, "rb") as f:
        return pickle.load(f)


In [38]:
# ---------- 8) Example usage ----------
if __name__ == "__main__":
    # CONFIGURE these:
    BASE_FOLDER = "Datasets/Unstructured_Data"     # <-- change to your folder
    EMBED_MODEL_NAME = "silma-ai/silma-embeddding-sts-v0.1"  # or other Arabic embedding model
    OUT_PATH = "./output/docs_and_embeddings.pkl"
    CHUNK_LONG_DOCS = True
    CHUNK_SIZE = 3000
    CHUNK_OVERLAP = 300
    EMBED_DEVICE = "cuda"  # change to "cuda" if available

    # Run pipeline
    out = build_document_index(
        BASE_FOLDER,
        embed_model_name=EMBED_MODEL_NAME,
        chunk_long_docs=CHUNK_LONG_DOCS,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        embed_device=EMBED_DEVICE,
    )

    # Save to disk
    save_index(out, OUT_PATH)

    # quick summary print
    print("\nSummary:")
    print(f" Documents: {len(out['documents'])}")
    print(f" Embeddings: {len(out['embeddings'])}")
    if out["embeddings"]:
        print(f" Vector dim: {len(out['embeddings'][0])}")

📂 Loading folder/category: 3okobat
✓ Loaded: 3okobat.pdf (chars: 266463)
✓ Loaded: 3okobat_num36_2020.txt (chars: 1261)
✓ Loaded: 3okobat_num6_2020.txt (chars: 1079)
📂 Loading folder/category: Asle7a
✓ Loaded: asle7a.txt (chars: 33032)
📂 Loading folder/category: child
✓ Loaded: child.txt (chars: 69170)
📂 Loading folder/category: dostor_gena2y
✓ Loaded: dostor.pdf (chars: 83841)
📂 Loading folder/category: drugs
✓ Loaded: drugs.txt (chars: 96062)
📂 Loading folder/category: egra2at_gena2ya
✓ Loaded: egra2at_gena2ya.pdf (chars: 178058)
📂 Loading folder/category: erhab
✓ Loaded: erhab.txt (chars: 33943)
📂 Loading folder/category: na2d
✓ Loaded: mbade2_ma7kmt_elna2d.pdf (chars: 331094)
📂 Loading folder/category: technology
✓ Loaded: tech.txt (chars: 29717)
✅ Total processed document objects (after chunking): 422
Loading embedding model: silma-ai/silma-embeddding-sts-v0.1 (device=cuda) ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1547.21it/s, Materializing param=pooler.dense.weight]                               


Model loaded.
🔢 Embedding texts ...


Batches: 100%|██████████| 14/14 [00:15<00:00,  1.08s/it]

✅ Embedded 422 texts (vectors length = 768)
Saved index to: ./output/docs_and_embeddings.pkl

Summary:
 Documents: 422
 Embeddings: 422
 Vector dim: 768


In [39]:
out['documents']

[Document(metadata={'source': '3okobat.pdf', 'path': 'Datasets/Unstructured_Data/3okobat/3okobat.pdf', 'category': '3okobat', 'chunk_index': 0, 'chunk_total': 99}, page_content='مركز معلومات النيابه العامه ــ مارس 2019 مركز معلومات النيابه العامه ــ مارس 2019 3 القانون رقم58 لسنه1937 باصدار قانون العقوبات وفقا الخر التعديالت لعام 2018 بالقانون رقم21 لسنه2018 مركز معلومات النيابه العامه ــ مارس 2019 4 حنن فاروق االول ملك مصر قرر جملس الشيوخ وجملس النواب القانون االتي نصه وقد صدقنا عليه واصدرناه : ( 1 ) املاده االويل من مواد االصدار يلغي قانون العقوبات الجاري العمل به امام املحاكم االهليه وقانون العقوبات الذي تطبقه املحاكم املختلطه ويستعاض عنهما بقانون العقوبات املرافق لهذا القانون. املاده الثانيه من مواد االصدار علي وزير الحقانيه تنفيذ هذا القانون ويعمل به من 15 اكتوبر سنه1937 . نامر بان يبصم هذا القانون بخاتم الدوله وان ينشر في الجريده الرسميه وينفذ كقانون من قوانين الدوله. صدر بسراي عابدي ن يف 22 مجادي االويل سنه1356 31 يوليو سنه1937 ــــــــــــــــــــــــــ ــــــــــــــــــــــــ